# PIANO — Quick Start

PIANO is both an integration method and a generative model. The `Composer` class is the main entry point.

## Minimal example

`adata.X` must hold raw counts.

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # Required by deterministic=True (default) on GPU; set before importing piano

import anndata as ad
from piano import Composer

adata = ad.read_h5ad("/path/to/data.h5ad")  # Raw counts in .X

pianist = Composer(
    adata=adata,
    categorical_covariate_keys=["batch", "donor"],
    hvg_batch_key="batch",
    max_epochs=200,
    outdir="./results/",
)

pianist.run_pipeline()

# Get integrated latent space
adata.obsm["X_piano"] = pianist.get_latent_representation()

# Get batch-corrected counts
corrected = pianist.get_counterfactual()   # marginal (default)

## Command-line script

Run from your terminal. Results are written to `--outdir`; add argparse arguments or edit the script directly to customise parameters. Note that the script's defaults differ from `Composer`'s: `--early_stopping` and `--deterministic` default to `False`.

```bash
python3 run_piano_integration.py \
  --version 0.0_piano_integration \
  --adata_train_list /path/to/adata.h5ad \
  --outdir ../results \
  --categorical_covariate_keys batch donor \
  --batch_key batch \
  --umap_labels cell_type
```

## Counterfactual generation

In [ ]:
# Marginal — batch correction (default)
corrected = pianist.get_counterfactual()

# Probabilistic reconstruction
recon = pianist.get_counterfactual(covariates=None)

# Custom covariates: keys must be covariates the model was trained with
custom = pianist.get_counterfactual(
    covariates={"batch": "batch1", "donor": "marginal"}
)

## Save and load

The pickle stores the `Composer` settings only; the trained weights are in `<outdir>/checkpoints/model_checkpoint.pt`.

In [ ]:
pianist.save("./results/pianist.pkl")

# Later, or in a new session
pianist = Composer.load(
    "./results/pianist.pkl",
    model_checkpoint_path="./results/checkpoints/model_checkpoint.pt",
    device="cuda",
)